# Notebook 6: Images in Topic Search (bonus)

Preprocess posts that are images using image-captioning with a vision model.

Goal: Understanding image-aware search retrieval

In [ ]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

## Outline
- Emeddings can be generated from multimodal content 
- Example of using a vision model to extract a caption and adding it to the
  document embedding. 

  Exercise: Update the preprocessing.py with reference code



## Handling Images

Recall the `PostDocument` from Notebook 2. Two fields matter here:

- `image_url` — path to an image file (or `None` for text-only posts).
- `image_caption` — a *text* caption produced by a vision model. Initially
  `None`.

We could use a multimodal model and embed the images and the text together.
However, for clarity and to allow more flexibility in what elements of the
PostDocument we include in the embeddings and are searchable, we separate them. 

1. Caption the image into text with a vision-language model (BLIP)
2. Use the same `all-MiniLM-L6-v2` embedding model to embed both the post text
   and the image caption. 
3. Store the combined embedding on the PostDocument. 

The combined embedding is used for search retrieval and topic modeling.

## Image Caption Pipeline

- **Retrieval embedding** *includes* the image caption. A post that is just a photo of
  a cat needs the caption "a cat sitting on a windowsill" to be findable when a user
  searches for *cat*.
- **Training embedding** *excludes* the image caption. If we train BERTopic on text +
  caption, the model can spuriously cluster posts by image-only artefacts (e.g.
  every photo at sunset → a fake "sunrise" topic) instead of what the post is
  actually *about*.

In this workshop we use the same string for both for simplicity, but the
`extract_embedding_text` seam in `src/preprocess.py` is where you'd split them.

### Step 1 — Load the BLIP vision model

Mirrors `PreprocessingPipeline._load_vision_model` in `src/preprocess.py:79`.

In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

from src.config import VISION_MODEL_NAME

# BLIP = "Bootstrapping Language-Image Pre-training". Two parts:
#   processor — turns a PIL image into a normalized 4-D tensor [batch, 3, 224, 224]
#   model     — generates caption token IDs autoregressively
processor = BlipProcessor.from_pretrained(
    VISION_MODEL_NAME,
    use_fast=False,           # fast image processor incompatible with this checkpoint
    use_fast_tokenizer=True,
)
model = BlipForConditionalGeneration.from_pretrained(
    VISION_MODEL_NAME,
    use_safetensors=True,
).to("cpu")
print("BLIP loaded:", VISION_MODEL_NAME)

### Step 2 — Caption a single image, end-to-end

Three calls: `processor` → `model.generate` → `processor.decode`.

1. The **processor** takes a PIL image, resizes it to 224×224, and normalizes pixel
   values into a `[1, 3, 224, 224]` tensor (1 image × 3 colour channels × H × W).
2. **`model.generate`** runs the image tensor through the vision encoder and
   autoregressively produces a sequence of caption token IDs.
3. **`processor.decode`** turns those IDs back into a human-readable string.

In [ ]:
from PIL import Image

from src.config import REPO

# Pick any image in assets/ — try cat1.jpg, space.jpg, fog.jpg, etc.
img_path = REPO / "assets" / "cat1.jpg"
image = Image.open(img_path).convert("RGB")
print(f"Loaded {img_path.name} — original size: {image.size}")

# 1) Pre-process the image into a tensor
inputs = processor(images=image, return_tensors="pt").to("cpu")
print("Processed tensor shape:", inputs["pixel_values"].shape, "  # [batch, RGB, H, W]")

# 2) Generate caption token IDs
output = model.generate(**inputs, max_new_tokens=256)
print("Token IDs shape:", output.shape)

# 3) Decode token IDs back to text
caption = processor.decode(output[0], skip_special_tokens=True)
print(f"\nCaption: {caption!r}")

### Try it on more images

Loop over a few of the sample assets to see how BLIP handles different scenes.

In [ ]:
for asset in ["cat1.jpg", "space.jpg", "fog.jpg", "food.jpg"]:
    img_path = REPO / "assets" / asset
    if not img_path.exists():
        print(f"  (skipped — {asset} not found)")
        continue
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to("cpu")
    output = model.generate(**inputs, max_new_tokens=256)
    caption = processor.decode(output[0], skip_special_tokens=True)
    print(f"  {asset:<14s} → {caption}")

# Exercise

Time: 3 minutes; 2-10 minutes to retrain the model.
Using what you learned above, add the code for the placeholder function that
captions images in the source code. 

File: 
`src/preprocess.py`  
Code: 
`PreprocessingPipeline()._caption_single_post()`


In [ ]:
# Reference solution for src.preprocess.PreprocessingPipeline._caption_single_post

def _caption_single_post(self, postdoc):
    """Run BLIP on one postdoc and set image_caption in-place."""
    if not postdoc.image_url:
        return
    img_path = REPO / postdoc.image_url
    if not img_path.exists():
        return
    image = Image.open(img_path).convert("RGB")

    # 1) Pre-process the image. Output shape: [1, 3, 224, 224]
    #    (one image · 3 RGB channels · 224×224 normalized pixels)
    inputs = self._vision_processor(images=image, return_tensors="pt").to("cpu")

    # 2) Generate caption token IDs
    output = self._vision_model.generate(**inputs, max_new_tokens=256)

    # 3) Decode tokens → string and attach to the postdoc
    caption = self._vision_processor.decode(output[0], skip_special_tokens=True)
    postdoc.image_caption = caption

print("Reference implementation — copy into src/preprocess.py.")

### Run the full pipeline

Once `_caption_single_post` is implemented, run the end-to-end pipeline so every
image-only post gets a caption, an embedding, and an Elasticsearch entry:

```bash
uv run python -m src.run_pipeline # or solutions.run_pipeline
```

Then refresh the demo app and search for **cat**. Image-only posts (no text!) should
now appear in the results, ranked by how cat-like their captions are.

### Optional: run from inside the notebook

If you prefer not to switch to a terminal, you can run the same preprocessing
pipeline here using your edited `src/run_pipeline.py`. 

Expect a 2-10 minute run to retrain th model

In [ ]:
# Run the full BERTtopic training pipeline.
from src.run_pipeline import pipeline
pipeline()